# Análise · Quatro bairros com queda fora da geração distribuída
### V-Energy Grid 360 — diagnóstico da receita

**Status: executado até a seção 10 (resultados nos textos de cada seção).**

Dá o **notebook de origem** às afirmações da seção 5.5 do `docs/diagnostico_receita.md`. Só lê tabelas já existentes (`gold.fato_consumo_faturamento`, `gold.dim_uc`, `gold.receita_decomposicao`) e **não grava nada**.

**Pergunta do Henrique:** "se foi volume, quero saber onde e por quê."

**Caminho:**
1. A `receita_decomposicao` mostra que **quatro bairros** (Iputinga e Macaxeira, em Recife; Jardim Jordão e Prazeres, em Jaboatão) respondem por 31,5% da perda de volume residencial.
2. Separando a perda das UCs **com** e **sem** GD, em 11 dos 15 bairros que mais perdem a GD explica tudo, e nesses quatro **63% da perda não vem da GD**.
3. A série mensal mostra uma **mudança de patamar** de cerca de −19%, entre maio e junho de 2025.
4. A distribuição entre as UCs mostra que a queda é **concentrada**: 21,7% das UCs sem GD desses bairros caíram 40% ou mais, contra 0,02% nos demais.
5. Sem GD, sem troca de medidor, sem mudança no tipo de leitura e sem alimentador comum, a causa **não aparecia nas tabelas de consumo**.
6. As seções 6 e 7 **a identificam**: 599 das 601 UCs foram inspecionadas, **91,7% tiveram irregularidade confirmada** (contra ~8% nas UCs sem queda), e **98,8% têm medidor eletromecânico** (contra ~23% na base).
7. A seção 8 mostra que **o consumo não voltou depois da inspeção**; a seção 9 mostra que as 387 inspeções fora desta análise têm **um segundo grupo, no comercial** (338 inspeções, 84,9% de acerto); a seção 10 aplica o mesmo teste ao comercial e mostra que **o grupo comercial está nos mesmos quatro bairros**: 300 UCs, 94,3% de irregularidades confirmadas, com a mesma data e o mesmo tamanho de queda (~55%).

**Cuidados:** é associação. O limiar de 40% é uma escolha nossa. A estimativa de energia usa a média mensal dos meses comparáveis (sem novembro e fevereiro) e multiplica por 12.

In [0]:
from pyspark.sql import functions as F

catalog = "v_energy"
gold_schema = f"{catalog}.gold"

# Os quatro bairros que concentram a queda sem GD (município, bairro)
QUATRO = [("Recife", "Iputinga"), ("Jaboatão dos Guararapes", "Jardim Jordão"),
          ("Recife", "Macaxeira"), ("Jaboatão dos Guararapes", "Prazeres")]
cond = F.lit(False)
for m, b in QUATRO:
    cond = cond | ((F.col("municipio") == m) & (F.col("bairro") == b))
grupo = F.when(cond, "quatro_bairros").otherwise("demais")

fato = spark.table(f"{gold_schema}.fato_consumo_faturamento").filter("classe = 'RESIDENCIAL'").withColumn("grupo", grupo)
dim = spark.table(f"{gold_schema}.dim_uc").filter("classe = 'RESIDENCIAL'")
rec = spark.table(f"{gold_schema}.receita_decomposicao").filter("classe = 'RESIDENCIAL'")
print("residencial na fato:", f"{fato.count():,}".replace(",", "."))

residencial na fato: 1.833.494


## 1. Os bairros que mais perderam volume residencial, com e sem GD

`perda_ucs_gd` é a variação do volume das UCs que têm GD no fim do período 1; `perda_sem_gd` é o resto. Esperado: nos quatro primeiros, `pct_perda_sem_gd` entre **−12,4% e −17,3%**; nos demais do topo, **positivo, de +3,6% a +4,6%**.

In [0]:
res = (
    rec.withColumn("perda_kwh", F.col("volume_kwh_1") - F.col("volume_kwh_0"))
    .withColumn("perda_ucs_gd", F.col("volume_kwh_ucs_com_gd_1") - F.col("volume_kwh_ucs_com_gd_0"))
    .withColumn("perda_sem_gd", F.col("perda_kwh") - F.col("perda_ucs_gd"))
    .withColumn("pct_perda_sem_gd", F.round(F.expr("try_divide(perda_sem_gd, volume_kwh_0 - volume_kwh_ucs_com_gd_0)") * 100, 1))
    .withColumn("pct_ucs_com_gd", F.round(F.expr("try_divide(qtd_ucs_com_gd_1, qtd_ucs_periodo_1)") * 100, 1))
)
display(res.select("municipio", "bairro", "perda_kwh", "perda_ucs_gd", "perda_sem_gd", "pct_perda_sem_gd",
                   "pct_ucs_com_gd", "qtd_trocas_medidor_1").orderBy("perda_kwh").limit(15))

# Quanto os quatro representam da perda residencial (esperado: 1.381.320 de 4.379.684 kWh = 31,5%; 62,6% sem GD)
tot = res.agg(F.sum("perda_kwh").alias("perda_total")).first()["perda_total"]
q = res.filter(cond).agg(F.sum("perda_kwh").alias("perda"), F.sum("perda_sem_gd").alias("sem_gd")).first()
print(f"perda residencial total: {tot} | quatro bairros: {q['perda']} ({q['perda'] / tot * 100:.1f}%) | sem GD: {q['sem_gd']} ({q['sem_gd'] / q['perda'] * 100:.1f}%)")

# UCs e trocas de medidor por 100 UCs: os quatro bairros contra os outros 11 dos 15 que mais perdem
# (esperado: quatro = 327 trocas em 3.025 UCs = 10,8 por 100; outros 11 = 1.003 trocas em 7.049 UCs = 14,2 por 100)
top15 = res.orderBy("perda_kwh").limit(15).withColumn("grupo_top", F.when(cond, "quatro bairros").otherwise("outros 11 do topo"))
display(top15.groupBy("grupo_top").agg(
    F.count("*").alias("bairros"), F.sum("qtd_ucs_periodo_1").alias("ucs"), F.sum("qtd_trocas_medidor_1").alias("trocas"),
    F.round(F.sum("qtd_trocas_medidor_1") / F.sum("qtd_ucs_periodo_1") * 100, 1).alias("trocas_por_100_ucs")))

municipio,bairro,perda_kwh,perda_ucs_gd,perda_sem_gd,pct_perda_sem_gd,pct_ucs_com_gd,qtd_trocas_medidor_1
Recife,Iputinga,-496099.00,-197649.00,-298450.00,-15.6,9.2,113
Jaboatão dos Guararapes,Jardim Jordão,-439331.00,-163415.00,-275916.00,-15.7,8.0,89
Recife,Macaxeira,-232678.00,-69401.00,-163277.00,-17.3,6.7,54
Jaboatão dos Guararapes,Prazeres,-213212.00,-86170.00,-127042.00,-12.4,7.6,71
Recife,Brasília Teimosa,-161506.00,-298948.00,137442.00,4.1,8.1,249
Jaboatão dos Guararapes,Barra de Jangada,-76744.00,-124033.00,47289.00,4.5,10.1,84
São Lourenço da Mata,Pixete,-68433.00,-117515.00,49082.00,4.5,9.1,82
Paulista,Fragoso,-67489.00,-133017.00,65528.00,4.6,8.3,93
Igarassu,Saramandaia,-62795.00,-89014.00,26219.00,3.6,11.2,70
Jaboatão dos Guararapes,Sucupira,-61336.00,-101887.00,40551.00,4.1,7.7,75


perda residencial total: -4379684.00 | quatro bairros: -1381320.00 (31.5%) | sem GD: -864685.00 (62.6%)


grupo_top,bairros,ucs,trocas,trocas_por_100_ucs
quatro bairros,4,3025,327,10.8
outros 11 do topo,11,7049,1003,14.2


## 2. A série mensal: uma mudança de patamar

Consumo médio por UC e mês, **residencial, só meses sem GD**. Esperado: a razão quatro bairros ÷ demais fica entre **1,005 e 1,011** de 2022-07 a 2025-04, cai para **0,914** em 2025-05 e fica entre **0,813 e 0,822** de 2025-06 a 2026-06 (consumo médio por UC no período 0 → período 1: demais 178,7 → 176,8 kWh, quatro bairros 180,2 → 144,4).

In [0]:
serie = (fato.filter("NOT possui_gd_no_mes")
         .groupBy("mes_referencia").pivot("grupo", ["demais", "quatro_bairros"])
         .agg(F.round(F.avg("consumo_kwh"), 1))
         .withColumn("razao", F.round(F.col("quatro_bairros") / F.col("demais"), 3))
         .orderBy("mes_referencia"))
serie.show(60)

# Consumo médio por UC e mês em cada período (média sobre os UC-meses, sem meses com GD)
# esperado, pelo diagnóstico (que usou a média das médias mensais): demais 178,7 -> 176,8 (-1,0%); quatro bairros 180,2 -> 144,4 (-19,9%)
# Se esta saída diferir na primeira casa decimal, o diagnóstico passa a usar os números daqui.
(fato.filter("NOT possui_gd_no_mes AND periodo_diagnostico IN ('PRIMEIRO_ANO', 'ULTIMOS_12M')")
     .groupBy("grupo").pivot("periodo_diagnostico", ["PRIMEIRO_ANO", "ULTIMOS_12M"]).agg(F.round(F.avg("consumo_kwh"), 1))
     .withColumn("variacao_pct", F.round((F.col("ULTIMOS_12M") / F.col("PRIMEIRO_ANO") - 1) * 100, 1))
     .show())

+--------------+------+--------------+-----+
|mes_referencia|demais|quatro_bairros|razao|
+--------------+------+--------------+-----+
|       2022-07| 100.7|         101.8|1.011|
|       2022-08| 101.5|         102.3|1.008|
|       2022-09| 123.5|         124.6|1.009|
|       2022-10| 169.9|         171.4|1.009|
|       2022-11| 210.7|         212.1|1.007|
|       2022-12| 225.9|         227.9|1.009|
|       2023-01| 240.6|         243.0| 1.01|
|       2023-02| 229.1|         231.0|1.008|
|       2023-03| 232.4|         234.4|1.009|
|       2023-04| 211.4|         212.8|1.007|
|       2023-05| 171.0|         172.7| 1.01|
|       2023-06| 127.3|         128.5|1.009|
|       2023-07| 100.7|         101.4|1.007|
|       2023-08| 101.5|         102.1|1.006|
|       2023-09| 123.5|         124.4|1.007|
|       2023-10| 169.8|         170.9|1.006|
|       2023-11| 210.5|         212.0|1.007|
|       2023-12| 225.7|         227.4|1.008|
|       2024-01| 240.0|         241.8|1.008|
|       20

## 3. A queda é concentrada, e não uniforme

Variação contra o mesmo mês do ano anterior, UCs sem GD, nos meses sem o salto de novembro e fevereiro. Esperado: **demais** `[-12,0; -6,5; 0,0; 7,0; 13,6]` e **quatro bairros** `[-56,0; -15,7; -3,5; 4,7; 12,1]` (p10, p25, p50, p75, p90). Só a ponta de baixo difere: o p90 é parecido (+12,1% e +13,6%), então não é uma queda de todas as UCs.

In [0]:
a = fato.filter("NOT possui_gd_no_mes AND consumo_kwh_ano_anterior IS NOT NULL "
                "AND mes_referencia IN ('2025-12','2026-01','2026-03','2026-04')")
(a.groupBy("grupo").agg(F.count("*").alias("linhas"),
     F.expr("percentile_approx(cast(variacao_consumo_ano_anterior_pct as double), array(0.1,0.25,0.5,0.75,0.9))").alias("p10_p25_p50_p75_p90"))
  .show(truncate=False))

+--------------+------+------------------------------------+
|grupo         |linhas|p10_p25_p50_p75_p90                 |
+--------------+------+------------------------------------+
|demais        |130159|[-12.0, -6.47, 0.0, 6.96, 13.61]    |
|quatro_bairros|11155 |[-56.03, -15.69, -3.52, 4.66, 12.12]|
+--------------+------+------------------------------------+



## 4. O que a queda não é

**Tipo de leitura e correção de consumo.** Esperado: a parcela de leituras não reais fica estável de 2025-03 a 2025-08 (~23% nos quatro bairros, ~18% nos demais, **já antes do degrau**), e o consumo corrigido é ~0,1% nos dois grupos.

**Alimentadores.** Esperado: seis alimentadores (`ALM-REC-024`, `ALM-JAB-007`, `ALM-REC-027`, `ALM-REC-028`, `ALM-JAB-013` e `ALM-JAB-014`), nenhum comum aos quatro bairros e nenhum dos três do evento de julho de 2025 (`ALM-REC-008`, `ALM-SAO-004` e `ALM-JAB-004`).

**Troca de medidor.** As trocas ocorrem de setembro de 2025 a março de 2026, depois do degrau (a seção 1 mostra ~10,8 trocas por 100 UCs nos quatro bairros, contra ~14,2 nos demais do topo).

In [0]:
(fato.filter("mes_referencia BETWEEN '2025-03' AND '2025-08'")
     .groupBy("mes_referencia", "grupo")
     .agg(F.round(F.avg((F.col("tipo_leitura") != "REAL").cast("int")) * 100, 1).alias("pct_leitura_nao_real"),
          F.round(F.avg(F.col("consumo_corrigido").cast("int")) * 100, 1).alias("pct_consumo_corrigido"))
     .orderBy("mes_referencia", "grupo").show(20))

(dim.filter(cond).groupBy("municipio", "bairro", "alimentador").count()
    .orderBy("municipio", "bairro", F.desc("count")).show(40, truncate=False))

+--------------+--------------+--------------------+---------------------+
|mes_referencia|         grupo|pct_leitura_nao_real|pct_consumo_corrigido|
+--------------+--------------+--------------------+---------------------+
|       2025-03|        demais|                18.6|                  0.1|
|       2025-03|quatro_bairros|                24.5|                  0.1|
|       2025-04|        demais|                18.3|                  0.1|
|       2025-04|quatro_bairros|                22.8|                  0.0|
|       2025-05|        demais|                18.6|                  0.1|
|       2025-05|quatro_bairros|                23.0|                  0.1|
|       2025-06|        demais|                18.2|                  0.1|
|       2025-06|quatro_bairros|                22.7|                  0.1|
|       2025-07|        demais|                18.4|                  0.1|
|       2025-07|quatro_bairros|                23.4|                  0.1|
|       2025-08|        d

## 5. Quantas UCs e quanta energia

Compara cada UC **com ela mesma**: média do consumo dos meses de `2024-07` a `2025-04` (sem novembro e fevereiro) contra os mesmos meses de `2025-07` a `2026-04`. Conta as UCs **sem GD** que caíram **40% ou mais**.

Esperado: **demais** 32.370 UCs, 7 com queda forte (0,02%); **quatro bairros** 2.774 UCs, **601 (21,7%)**, mediana da queda das afetadas de **−55,1%**, e **1.059.594 kWh por ano** (média mensal da diferença × 12).

A 40% é uma escolha nossa. O grupo `demais` mostra o ruído de fundo: 7 UCs em 32 mil.

In [0]:
meses_depois = ['2025-07', '2025-08', '2025-09', '2025-10', '2025-12', '2026-01', '2026-03', '2026-04']
meses_antes  = ['2024-07', '2024-08', '2024-09', '2024-10', '2024-12', '2025-01', '2025-03', '2025-04']
ucs_sem_gd = dim.filter("NOT possui_gd").select("id_uc")

r = (fato.join(ucs_sem_gd, "id_uc", "left_semi")
     .withColumn("periodo", F.when(F.col("mes_referencia").isin(meses_antes), "antes")
                             .when(F.col("mes_referencia").isin(meses_depois), "depois"))
     .filter("periodo IS NOT NULL")
     .groupBy("id_uc", "grupo", "periodo").agg(F.avg("consumo_kwh").alias("media"))
     .groupBy("id_uc", "grupo").pivot("periodo", ["antes", "depois"]).agg(F.first("media"))
     .filter("antes > 0 AND depois IS NOT NULL")
     .withColumn("queda_pct", (F.col("depois") / F.col("antes") - 1) * 100)
     .withColumn("queda_forte", F.col("queda_pct") <= -40))

(r.groupBy("grupo").agg(
    F.count("*").alias("ucs"),
    F.sum(F.col("queda_forte").cast("int")).alias("ucs_queda_40"),
    F.round(F.avg(F.col("queda_forte").cast("int")) * 100, 1).alias("pct_ucs_queda_40"),
    F.expr("percentile_approx(CASE WHEN queda_forte THEN queda_pct END, 0.5)").alias("mediana_queda_das_afetadas"),
    F.round(F.sum(F.when(F.col("queda_forte"), F.col("antes") - F.col("depois"))) * 12, 0).alias("kwh_ano_perdidos"))
  .show())

+--------------+-----+------------+----------------+--------------------------+----------------+
|         grupo|  ucs|ucs_queda_40|pct_ucs_queda_40|mediana_queda_das_afetadas|kwh_ano_perdidos|
+--------------+-----+------------+----------------+--------------------------+----------------+
|        demais|32370|           7|             0.0|         -41.4387031408308|          8429.0|
|quatro_bairros| 2774|         601|            21.7|       -55.112881806108895|       1059594.0|
+--------------+-----+------------+----------------+--------------------------+----------------+



## 6. O que as inspeções dizem

Cruza as UCs desta análise com a `silver.inspecoes` (1.389 inspeções de agosto a dezembro de 2025; 871 com irregularidade confirmada, **62,7%**). Cada inspeção cai em uma categoria, conforme a UC:

1. **Quatro bairros, queda de 40% ou mais** (as 601 UCs);
2. quatro bairros, sem queda forte;
3. demais bairros, queda de 40% ou mais (as 7 do ruído de fundo);
4. demais bairros, sem queda forte;
5. fora da análise (UC com GD, de outra classe ou sem dados nos meses comparados).

**Como ler o resultado** (todos os números absolutos importam: uma taxa com poucas inspeções não prova nada):
- **Muitas inspeções na categoria 1 com taxa de acerto bem acima das categorias 2 e 4** (e acima de 62,7%): fortalece a hipótese de irregularidade, com a ressalva de que a equipe escolhe as UCs que inspeciona.
- **Taxa parecida ou menor:** enfraquece a hipótese. A queda teria outra causa.
- **Poucas inspeções na categoria 1:** inconclusivo. O diagnóstico continua dizendo "causa não identificada".

**O sinal geral.** A última tabela da seção mede a taxa de acerto por faixa de queda de consumo em **todas** as UCs inspecionadas desta análise, e não só nos quatro bairros. Responde à pergunta da `fila_inspecao`: uma queda forte de consumo (sem GD) prevê irregularidade? Atenção: se a inspeção corrigiu a irregularidade, o consumo posterior sobe, e isso atenua a queda medida.

**Resultado da execução:**

| Categoria | Inspeções | Confirmadas | Taxa | kWh recuperados (média das confirmadas) |
|---|---|---|---|---|
| 1. quatro bairros, queda >= 40% | 599 (599 UCs) | 549 | **91,7%** | 1.011 |
| 2. quatro bairros, sem queda forte | 23 | 2 | 8,7% | 162 |
| 4. demais bairros, sem queda forte | 380 | 29 | 7,6% | 267 |
| 5. fora da análise (GD, outra classe ou sem dados) | 387 | 291 | 75,2% | 3.127 |

Não há inspeções na categoria 3 (as 7 UCs do ruído de fundo). Tipos nas 549 confirmadas da categoria 1: `MEDIDOR ADULTERADO` 119, `DESVIO DIRETO` 117, `BY-PASS` 116, `LIGACAO CLANDESTINA` 105 e `LACRE VIOLADO` 92. Por faixa de queda: **40% ou mais, 91,7% (599 inspeções)**; 15% a 40%, 0% (2); estável, **7,7% (401)**.

In [0]:
silver_schema = f"{catalog}.silver"
insp = spark.table(f"{silver_schema}.inspecoes")
print("inspeções:", insp.count(), "| confirmadas:", insp.filter("irregularidade_confirmada").count())   # esperado: 1.389 | 871 (62,7%)

# r (seção 5): UCs residenciais sem GD, com a queda de cada uma
i = (insp.join(r.select("id_uc", "grupo", "queda_pct", "queda_forte"), "id_uc", "left")
     .withColumn("categoria",
        F.when(F.col("grupo").isNull(), "5. fora da análise (GD, outra classe ou sem dados)")
         .when((F.col("grupo") == "quatro_bairros") & F.col("queda_forte"), "1. quatro bairros, queda >= 40%")
         .when(F.col("grupo") == "quatro_bairros", "2. quatro bairros, sem queda forte")
         .when(F.col("queda_forte"), "3. demais bairros, queda >= 40%")
         .otherwise("4. demais bairros, sem queda forte")))

display(i.groupBy("categoria").agg(
    F.count("*").alias("inspecoes"),
    F.countDistinct("id_uc").alias("ucs_inspecionadas"),
    F.sum(F.col("irregularidade_confirmada").cast("int")).alias("confirmadas"),
    F.round(F.avg(F.col("irregularidade_confirmada").cast("int")) * 100, 1).alias("pct_confirmadas"),
    F.round(F.avg(F.when(F.col("irregularidade_confirmada"), F.col("kwh_recuperado").cast("double"))), 0).alias("kwh_recuperado_medio_das_confirmadas"),
).orderBy("categoria"))

# Tipos de irregularidade e meses de inspeção das UCs da categoria 1
cat1 = i.filter(F.col("categoria").startswith("1."))
display(cat1.groupBy("tipo_irregularidade").count().orderBy(F.desc("count")))
display(cat1.groupBy(F.date_format("data_inspecao", "yyyy-MM").alias("mes_inspecao"))
        .agg(F.count("*").alias("inspecoes"), F.sum(F.col("irregularidade_confirmada").cast("int")).alias("confirmadas"))
        .orderBy("mes_inspecao"))

# O sinal geral: a taxa de acerto por faixa de queda de consumo, em todas as UCs inspecionadas desta análise
faixas = (i.filter(F.col("grupo").isNotNull())
          .withColumn("faixa", F.when(F.col("queda_pct") <= -40, "a) queda de 40% ou mais")
                                .when(F.col("queda_pct") <= -15, "b) queda de 15% a 40%")
                                .when(F.col("queda_pct") < 15, "c) estável (±15%)")
                                .otherwise("d) aumento de 15% ou mais")))
display(faixas.groupBy("faixa").agg(
    F.count("*").alias("inspecoes"),
    F.sum(F.col("irregularidade_confirmada").cast("int")).alias("confirmadas"),
    F.round(F.avg(F.col("irregularidade_confirmada").cast("int")) * 100, 1).alias("pct_confirmadas")).orderBy("faixa"))

inspeções: 1389 | confirmadas: 871


categoria,inspecoes,ucs_inspecionadas,confirmadas,pct_confirmadas,kwh_recuperado_medio_das_confirmadas
"1. quatro bairros, queda >= 40%",599,599,549,91.7,1011.0
"2. quatro bairros, sem queda forte",23,23,2,8.7,162.0
"4. demais bairros, sem queda forte",380,380,29,7.6,267.0
"5. fora da análise (GD, outra classe ou sem dados)",387,387,291,75.2,3127.0


tipo_irregularidade,count
MEDIDOR ADULTERADO,119
DESVIO DIRETO,117
BY-PASS,116
LIGACAO CLANDESTINA,105
LACRE VIOLADO,92
null,50


mes_inspecao,inspecoes,confirmadas
2025-08,112,103
2025-09,129,123
2025-10,126,117
2025-11,116,102
2025-12,116,104


faixa,inspecoes,confirmadas,pct_confirmadas
a) queda de 40% ou mais,599,549,91.7
b) queda de 15% a 40%,2,0,0.0
c) estável (±15%),401,31,7.7


## 7. Perfil do medidor das UCs afetadas

Compara as 601 UCs com as outras dos quatro bairros e com as dos demais bairros: tipo e idade do medidor ativo (`dim_uc`) e a parcela de UCs com troca recente. Um perfil diferente (por exemplo, quase todas com medidor eletromecânico, ou todas com medidor de um mesmo modelo) seria uma pista sobre a causa. Perfil igual descarta o medidor como explicação.

**Resultado da execução:** as 601 UCs têm **98,8% de medidores eletromecânicos**, idade média de **15,9 anos**, 1,0% de medidores `SMART` e 1,0% com troca recente. As outras UCs dos quatro bairros: 23,8%, 14,1 anos, 18,9% e 13,8%; os demais bairros: 22,5%, 14,3 anos, 19,2% e 14,4%.

In [0]:
perfil = (dim.join(r.select("id_uc", "grupo", "queda_forte"), "id_uc")
          .withColumn("categoria",
              F.when((F.col("grupo") == "quatro_bairros") & F.col("queda_forte"), "1. quatro bairros, queda >= 40%")
               .when(F.col("grupo") == "quatro_bairros", "2. quatro bairros, sem queda forte")
               .otherwise("3. demais bairros")))
display(perfil.groupBy("categoria").agg(
    F.count("*").alias("ucs"),
    F.round(F.avg(F.col("idade_medidor_anos").cast("double")), 1).alias("idade_media_medidor_anos"),
    F.round(F.avg((F.col("tipo_medidor_ativo") == "ELETROMECANICO").cast("int")) * 100, 1).alias("pct_eletromecanico"),
    F.round(F.avg((F.col("tipo_medidor_ativo") == "SMART").cast("int")) * 100, 1).alias("pct_smart"),
    F.round(F.avg(F.col("medidor_trocado_recentemente").cast("int")) * 100, 1).alias("pct_medidor_trocado")).orderBy("categoria"))

categoria,ucs,idade_media_medidor_anos,pct_eletromecanico,pct_smart,pct_medidor_trocado
"1. quatro bairros, queda >= 40%",601,15.9,98.8,1.0,1.0
"2. quatro bairros, sem queda forte",2173,14.1,23.8,18.9,13.8
3. demais bairros,32370,14.3,22.5,19.2,14.4


## 8. O consumo voltou depois da inspeção?

As 549 UCs com irregularidade confirmada foram inspecionadas entre agosto e dezembro de 2025. Se a inspeção regularizou a medição, o consumo delas deveria **voltar a subir** nos meses seguintes. A série mensal da seção 2 sugere que **não voltou** (a razão fica em 0,81 a 0,82 até junho de 2026), mas ela mistura as 601 com os demais moradores dos quatro bairros.

Esta seção mede só as 549 UCs, em **meses relativos à inspeção** (0 = mês da inspeção), como **razão ao consumo médio dos demais bairros no mesmo mês** (para descontar a sazonalidade). Se houve regularização, a razão sobe depois do mês 0; se continua baixa, não há sinal de recuperação no consumo faturado.

**Cuidados:** a razão das afetadas já era menor que 1 antes da inspeção (a queda é de ~55%). O que importa é a **variação** depois do mês 0. Não sabemos o que ocorreu após a confirmação (troca do medidor, cobrança retroativa, religação): os dados só mostram o consumo faturado.

**Resultado da execução** (razão ao consumo dos demais bairros, nas 549 UCs): **1,34** seis meses antes da inspeção, **0,98** quatro meses antes, **0,72** de dois meses antes até o mês da inspeção, e **0,72 a 0,73 nos oito meses seguintes** (445 UCs no mês 7 e 343 no mês 8, porque a base termina em 2026-06). **Não há recuperação do consumo faturado depois da inspeção.**

In [0]:
conf = (insp.join(r.filter("grupo = 'quatro_bairros' AND queda_forte").select("id_uc"), "id_uc", "left_semi")
        .filter("irregularidade_confirmada").select("id_uc", "data_inspecao"))
print("UCs confirmadas na categoria 1:", conf.select("id_uc").distinct().count())    # esperado: 549

demais_mes = (fato.filter("grupo = 'demais' AND NOT possui_gd_no_mes")
              .groupBy("mes_referencia").agg(F.avg("consumo_kwh").alias("dem_medio")))

rel = (fato.join(conf, "id_uc").join(demais_mes, "mes_referencia")
       .withColumn("mes_rel", F.months_between(F.to_date(F.concat("mes_referencia", F.lit("-01"))),
                                                F.trunc("data_inspecao", "month")).cast("int"))
       .filter("mes_rel BETWEEN -6 AND 8")
       .withColumn("razao", F.col("consumo_kwh") / F.col("dem_medio")))

display(rel.groupBy("mes_rel").agg(F.count("*").alias("ucs_mes"), F.round(F.avg("razao"), 3).alias("razao_media")).orderBy("mes_rel"))

UCs confirmadas na categoria 1: 549


mes_rel,ucs_mes,razao_media
-6,549,1.337
-5,549,1.171
-4,549,0.982
-3,549,0.807
-2,549,0.717
-1,549,0.722
0,549,0.721
1,549,0.72
2,549,0.722
3,549,0.722


## 9. As 387 inspeções fora desta análise

A categoria 5 reúne as inspeções de UCs que não entraram no estudo (com GD, de outra classe ou sem dados nos meses comparados). Ela tem **291 irregularidades confirmadas (75,2%)** e uma energia recuperada média de **3.127 kWh**, três vezes a da categoria 1. Esta seção separa por classe e por GD para saber **onde estão** essas irregularidades, que a `fila_inspecao` também precisa encontrar.

**Resultado da execução:**

| Classe | Com GD | Inspeções | Confirmadas | Taxa | kWh recuperados (média) |
|---|---|---|---|---|---|
| COMERCIAL | não | **338** | **287** | **84,9%** | 3.166 |
| RESIDENCIAL | sim | 28 | 1 | 3,6% | 143 |
| INDUSTRIAL | não | 10 | 1 | 10,0% | 488 |
| RURAL | não | 4 | 1 | 25,0% | 306 |
| COMERCIAL | sim | 3 | 0 | 0% | |
| RESIDENCIAL | não (sem dados nos meses comparados) | 2 | 1 | 50,0% | 479 |
| PODER PÚBLICO | não | 2 | 0 | 0% | |

Há **um segundo grupo: UCs comerciais sem GD**, com 84,9% de acerto e o triplo da energia recuperada. Somados, os dois grupos (549 residenciais e 287 comerciais) respondem por **836 das 871 irregularidades confirmadas (96%)**. As inspeções em UCs **com GD** (31) confirmaram só 1 irregularidade: quando a queda é da GD, a inspeção quase nunca encontra nada.

In [0]:
fora = (i.filter(F.col("categoria").startswith("5."))
        .join(spark.table(f"{gold_schema}.dim_uc").select("id_uc", "classe", "possui_gd"), "id_uc", "left"))
display(fora.groupBy("classe", "possui_gd").agg(
    F.count("*").alias("inspecoes"),
    F.sum(F.col("irregularidade_confirmada").cast("int")).alias("confirmadas"),
    F.round(F.avg(F.col("irregularidade_confirmada").cast("int")) * 100, 1).alias("pct_confirmadas"),
    F.round(F.avg(F.when(F.col("irregularidade_confirmada"), F.col("kwh_recuperado").cast("double"))), 0).alias("kwh_recuperado_medio"))
  .orderBy(F.desc("inspecoes")))

classe,possui_gd,inspecoes,confirmadas,pct_confirmadas,kwh_recuperado_medio
COMERCIAL,false,338,287,84.9,3166.0
RESIDENCIAL,true,28,1,3.6,143.0
INDUSTRIAL,false,10,1,10.0,488.0
RURAL,false,4,1,25.0,306.0
COMERCIAL,true,3,0,0.0,null
RESIDENCIAL,false,2,1,50.0,479.0
PODER PUBLICO,false,2,0,0.0,null


## 10. O mesmo teste na classe comercial

A seção 9 mostra 338 inspeções em UCs comerciais sem GD, com 84,9% de irregularidades confirmadas. Esta seção repete, para o comercial, o que a seção 5 fez para o residencial: compara cada UC **sem GD** com ela mesma (os mesmos meses de `meses_antes` e `meses_depois`) e conta as que caíram **40% ou mais**. Responde a quatro perguntas:
1. Quantas UCs comerciais caíram, e quanta energia por ano?
2. **Onde** estão (município e bairro)?
3. **Quando** a queda começou (a série mensal das afetadas contra as demais)?
4. Uma queda forte de consumo prevê irregularidade também no comercial (a taxa de acerto por faixa)?

Se as UCs comerciais com queda forte forem as mesmas das 338 inspecionadas, o sinal da `fila_inspecao` vale para as duas classes.

**Resultado da execução:**
1. **300 de 4.518 UCs comerciais sem GD (6,6%)** caíram 40% ou mais, com queda mediana de 54%, cerca de **1.796.304 kWh por ano**.
2. **Estão nos mesmos quatro bairros:** Iputinga 90 de 107, Jardim Jordão 95 de 116, Prazeres 71 de 87 e Macaxeira 43 de 55 (78% a 84% das UCs comerciais de cada um); fora deles, 1 UC (em Caixa D'Água, Olinda).
3. **Mesmo mês:** a razão afetadas ÷ demais fica em 1,13 a 1,15 até 2025-04, cai para 0,823 em 2025-05 e fica em **0,51 a 0,52 de 2025-06 a 2026-06**.
4. **O sinal prevê irregularidade:** queda de 40% ou mais, 299 inspeções e 282 confirmadas (**94,3%**); estável, 37 inspeções e 3 confirmadas (8,1%).

In [0]:
fato_c = spark.table(f"{gold_schema}.fato_consumo_faturamento").filter("classe = 'COMERCIAL'")
ucs_c = spark.table(f"{gold_schema}.dim_uc").filter("classe = 'COMERCIAL' AND NOT possui_gd").select("id_uc")
base_c = fato_c.join(ucs_c, "id_uc", "left_semi")

rc = (base_c
      .withColumn("periodo", F.when(F.col("mes_referencia").isin(meses_antes), "antes")
                              .when(F.col("mes_referencia").isin(meses_depois), "depois"))
      .filter("periodo IS NOT NULL")
      .groupBy("id_uc", "municipio", "bairro", "periodo").agg(F.avg("consumo_kwh").alias("media"))
      .groupBy("id_uc", "municipio", "bairro").pivot("periodo", ["antes", "depois"]).agg(F.first("media"))
      .filter("antes > 0 AND depois IS NOT NULL")
      .withColumn("queda_pct", (F.col("depois") / F.col("antes") - 1) * 100)
      .withColumn("queda_forte", F.col("queda_pct") <= -40))

# 1. Quantas UCs e quanta energia
display(rc.agg(F.count("*").alias("ucs"),
               F.sum(F.col("queda_forte").cast("int")).alias("ucs_queda_40"),
               F.round(F.avg(F.col("queda_forte").cast("int")) * 100, 1).alias("pct_ucs_queda_40"),
               F.expr("percentile_approx(CASE WHEN queda_forte THEN queda_pct END, 0.5)").alias("mediana_queda_das_afetadas"),
               F.round(F.sum(F.when(F.col("queda_forte"), F.col("antes") - F.col("depois"))) * 12, 0).alias("kwh_ano_perdidos")))

# 2. Onde (os 15 bairros com mais UCs afetadas)
display(rc.groupBy("municipio", "bairro").agg(
    F.count("*").alias("ucs"), F.sum(F.col("queda_forte").cast("int")).alias("afetadas"),
    F.round(F.avg(F.col("queda_forte").cast("int")) * 100, 1).alias("pct_afetadas")).orderBy(F.desc("afetadas")).limit(15))

# 3. Quando: consumo médio por UC e mês das afetadas contra as demais UCs comerciais sem GD
serie_c = (base_c.join(rc.select("id_uc", "queda_forte"), "id_uc")
           .withColumn("g", F.when(F.col("queda_forte"), "afetadas").otherwise("demais"))
           .groupBy("mes_referencia").pivot("g", ["demais", "afetadas"]).agg(F.round(F.avg("consumo_kwh"), 1))
           .withColumn("razao", F.round(F.col("afetadas") / F.col("demais"), 3)).orderBy("mes_referencia"))
serie_c.show(60)

# 4. A queda prevê irregularidade no comercial? (taxa de acerto por faixa de queda, UCs inspecionadas)
ic = insp.join(rc.select("id_uc", "queda_pct"), "id_uc")
display(ic.withColumn("faixa", F.when(F.col("queda_pct") <= -40, "a) queda de 40% ou mais")
                               .when(F.col("queda_pct") <= -15, "b) queda de 15% a 40%")
                               .when(F.col("queda_pct") < 15, "c) estável (±15%)")
                               .otherwise("d) aumento de 15% ou mais"))
        .groupBy("faixa").agg(F.count("*").alias("inspecoes"),
                              F.sum(F.col("irregularidade_confirmada").cast("int")).alias("confirmadas"),
                              F.round(F.avg(F.col("irregularidade_confirmada").cast("int")) * 100, 1).alias("pct_confirmadas"))
        .orderBy("faixa"))

ucs,ucs_queda_40,pct_ucs_queda_40,mediana_queda_das_afetadas,kwh_ano_perdidos
4518,300,6.6,-54.42222649490483,1796304.0


municipio,bairro,ucs,afetadas,pct_afetadas
Jaboatão dos Guararapes,Jardim Jordão,116,95,81.9
Recife,Iputinga,107,90,84.1
Jaboatão dos Guararapes,Prazeres,87,71,81.6
Recife,Macaxeira,55,43,78.2
Olinda,Caixa D'Água,17,1,5.9
Recife,Água Fria,16,0,0.0
Ipojuca,Canoas,7,0,0.0
Recife,Rosarinho,45,0,0.0
Cabo de Santo Agostinho,Pirapama,59,0,0.0
Camaragibe,Vila da Fábrica,44,0,0.0


+--------------+------+--------+-----+
|mes_referencia|demais|afetadas|razao|
+--------------+------+--------+-----+
|       2022-07| 581.3|   664.0|1.142|
|       2022-08| 583.4|   667.7|1.144|
|       2022-09| 649.1|   738.5|1.138|
|       2022-10| 784.0|   890.0|1.135|
|       2022-11| 903.5|  1033.0|1.143|
|       2022-12| 950.2|  1088.8|1.146|
|       2023-01| 992.4|  1134.3|1.143|
|       2023-02| 958.6|  1092.3|1.139|
|       2023-03| 967.9|  1108.0|1.145|
|       2023-04| 905.0|  1039.0|1.148|
|       2023-05| 788.3|   894.6|1.135|
|       2023-06| 658.0|   750.3| 1.14|
|       2023-07| 579.2|   664.1|1.147|
|       2023-08| 582.4|   666.5|1.144|
|       2023-09| 646.2|   741.7|1.148|
|       2023-10| 782.2|   895.8|1.145|
|       2023-11| 904.2|  1041.2|1.152|
|       2023-12| 948.9|  1079.6|1.138|
|       2024-01| 991.6|  1135.9|1.146|
|       2024-02| 957.5|  1098.5|1.147|
|       2024-03| 965.0|  1103.4|1.143|
|       2024-04| 903.9|  1024.8|1.134|
|       2024-05| 789.0|  

faixa,inspecoes,confirmadas,pct_confirmadas
a) queda de 40% ou mais,299,282,94.3
c) estável (±15%),37,3,8.1
